#### Adaptive Hybrid RAG Pipeline
##### Data Ingestion and Importing important libraries

In [41]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datasets import load_dataset
from dotenv import load_dotenv
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer,CrossEncoder
import torch
from groq import Groq
from langchain_groq import ChatGroq
import re


#### Loading HF Token from HuggingFace for authenticated requests

In [2]:
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

#### Data Chunking and Embedding 
###### (npy and json file downloaded from google colab with runtime set to T4 GPU due to laptop's CPU constraints(lack of GPU = 26hrs+ to load all 10000+ documents))

In [3]:
"""
import re
import hashlib
dataset = load_dataset("ccdv/arxiv-summarization")
paper = dataset["train"][0]
print(paper.keys())
subset = dataset["train"].select(range(10000))
subset
def clean_text(text):
    text = re.sub(r"@xcite", "", text)       
    text = re.sub(r"@xmath\d+", "[MATH]", text)  
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()
def process_dataset(data):
    all_chunks = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    for i, paper in enumerate(data):
        print(f"[INFO] Processing document {i}...")
        try:
            article = clean_text(paper.get("article", ""))
            abstract = clean_text(paper.get("abstract", ""))
            abstract_chunks = splitter.split_text(abstract)
            for chunk_idx, chunk in enumerate(abstract_chunks):
                chunk = chunk.strip()
                if len(chunk) < 50:
                    continue
                chunk_id = hashlib.md5(
                    f"{i}_abstract_{chunk_idx}".encode()
                ).hexdigest()
                all_chunks.append({
                    "chunk_id": chunk_id,
                    "paper_id": i,
                    "section_title": "ABSTRACT",
                    "chunk_index": chunk_idx,
                    "text": f"{abstract[:200]} {chunk}",
                    "abstract": abstract[:300],
                    "token_estimate": len(chunk.split())
                })
            article_chunks = splitter.split_text(article)
            for chunk_idx, chunk in enumerate(article_chunks):
                chunk = chunk.strip()
                if len(chunk) < 50:
                    continue
                chunk_id = hashlib.md5(
                    f"{i}_article_{chunk_idx}".encode()
                ).hexdigest()
                all_chunks.append({
                    "chunk_id": chunk_id,
                    "paper_id": i,
                    "section_title": "ARTICLE",
                    "chunk_index": chunk_idx,
                    "text": chunk,
                    "abstract": abstract[:300],
                    "token_estimate": len(chunk.split())
                })

        except Exception as e:
            print(f"[ERROR] Failed processing document {i}: {e}")

    print(f"\n[INFO] Successfully created {len(all_chunks)} chunks from {len(data)} papers")
    return all_chunks
all_chunks = process_dataset(subset)
all_chunks
"""

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\mishr\AppData\Local\Temp\ipykernel_9072\1794663228.py:11: SyntaxWarning: invalid escape sequence '\d'
  text = re.sub(r"@xmath\d+", "[MATH]", text)


'\nimport re\nimport hashlib\ndataset = load_dataset("ccdv/arxiv-summarization")\npaper = dataset["train"][0]\nprint(paper.keys())\nsubset = dataset["train"].select(range(10000))\nsubset\ndef clean_text(text):\n    text = re.sub(r"@xcite", "", text)       \n    text = re.sub(r"@xmath\\d+", "[MATH]", text)  \n    text = re.sub(r"[ \t]+", " ", text)\n    text = re.sub(r"\n{3,}", "\n\n", text)\n    return text.strip()\ndef process_dataset(data):\n    all_chunks = []\n    splitter = RecursiveCharacterTextSplitter(\n        chunk_size=1000,\n        chunk_overlap=200,\n        separators=["\n\n", "\n", ". ", " ", ""]\n    )\n    for i, paper in enumerate(data):\n        print(f"[INFO] Processing document {i}...")\n        try:\n            article = clean_text(paper.get("article", ""))\n            abstract = clean_text(paper.get("abstract", ""))\n            abstract_chunks = splitter.split_text(abstract)\n            for chunk_idx, chunk in enumerate(abstract_chunks):\n                chu

#### Loading relevant files downloaded from google colab

In [13]:
import json
import numpy as np
with open("../all_chunks_slim.json") as f:
    all_chunks = json.load(f)
chunk_embeddings = np.load("../chunk_embeddings_10k (1).npy")
print(f"Chunks: {len(all_chunks)}")
print(f"Embeddings: {chunk_embeddings.shape}")

Chunks: 443080
Embeddings: (443080, 384)


#### Checking the contents of the 10000+ files

In [5]:
import random
from collections import Counter
import re
random.seed(42)
sample_chunks = random.sample(all_chunks, min(300, len(all_chunks)))
word_counts = Counter()
stopwords = {
    "the","a","an","of","in","to","and","is","are","for","with","that",
    "this","on","by","we","be","as","from","it","or","at","our","their",
    "which","have","has","been","can","also","these","they","its","was",
    "were","not","but","such","using","used","when","where","all","each",
    "two","one","two","three","more","than","into","over","between","after",
    "both","while","than","thus","well","here","then","there","so","may",
    "will","would","could","should","about","other","through","however",
    "paper","model","show","shows","method","approach","results","result",
    "based","proposed","present","use","data","set","number","given"
}
for chunk in sample_chunks:
    words = re.findall(r'\b[a-z]{4,}\b', chunk["text"].lower())
    for w in words:
        if w not in stopwords:
            word_counts[w] += 1
print("═" * 60)
print("TOP 30 DOMAIN WORDS IN YOUR CORPUS (stopwords removed)")
print("═" * 60)
for word, count in word_counts.most_common(30):
    bar = "█" * (count // 5)
    print(f"  {word:<20} {count:>4}  {bar}")
print()
print(f"Total chunks in corpus : {len(all_chunks):,}")
print(f"Sample size            : {len(sample_chunks)}")
print()


════════════════════════════════════════════════════════════
TOP 30 DOMAIN WORDS IN YOUR CORPUS (stopwords removed)
════════════════════════════════════════════════════════════
  math                 2426  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  field                  78  ███████████████
  time                   77  ███████████████
  case                   64  ████████████
  state                  64  ████████████
  energy                 61  ████████████
  order                  58  ███████████
  function               56  ███████████
  system  

#### Run once to initialise BM25 index, save it to bm25_index_10k

In [ ]:
import bm25s
corpus_texts = [c["text"] for c in all_chunks]
retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(corpus_texts))
retriever.save("bm25_index_10k")
print(f"[INFO]BM25 index saved")

Split strings:  69%|██████▊   | 304183/443080 [00:18<00:07, 17657.93it/s]

#### Once bm25s index is created simply load it

In [1]:
import bm25s
retriever = bm25s.BM25.load("bm25_index_10k", load_corpus=True)
print(f"[INFO]BM25 loaded from disk")

c:\Users\mishr\OneDrive\Desktop\adaptive-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO]BM25 loaded from disk


#### Initialise QDrant Client for VectorDB storage

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
qdrant = QdrantClient(path="./qdrant_database_10k") 
qdrant.recreate_collection(
    collection_name="papers",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

C:\Users\mishr\AppData\Local\Temp\ipykernel_17912\359453757.py:3: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <papers> contains 443080 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant = QdrantClient(path="./qdrant_database_10k")
C:\Users\mishr\AppData\Local\Temp\ipykernel_17912\359453757.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

#### Run once to create collections Note: replace recreate_collection with create_collection

In [8]:
qdrant.recreate_collection(
    collection_name="papers",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)
BATCH = 512
for start in range(0, len(all_chunks), BATCH):
    end = min(start + BATCH, len(all_chunks))
    points = [
        PointStruct(
            id=i,
            vector=chunk_embeddings[i].tolist(),
            payload={
                "chunk_id": all_chunks[i]["chunk_id"],
                "paper_id": all_chunks[i]["paper_id"],
                "section": all_chunks[i]["section_title"],
                "text": all_chunks[i]["text"]
            }
        )
        for i in range(start, end)
    ]
    qdrant.upsert(collection_name="papers", points=points)
    if start % 50000 == 0:
        print(f"Indexed {start}/{len(all_chunks)}")
print(f"[INFO]Qdrant indexing complete")

C:\Users\mishr\AppData\Local\Temp\ipykernel_7008\16691855.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


Indexed 0/443080


C:\Users\mishr\AppData\Local\Temp\ipykernel_7008\16691855.py:21: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(collection_name="papers", points=points)


[INFO]Qdrant indexing complete


#### Sanity check to see if qdrant vectorized all the chunks->expected output : 443080

In [3]:
info = qdrant.get_collection("papers")
print(f"Total vectors in Qdrant: {info.points_count}")

Total vectors in Qdrant: 443080


#### Initialise all-MiniLM-L6-v2 Encoder

In [6]:
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("bi_encoder ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2010.29it/s]


bi_encoder ready


#### Initialize ms-marco-MiniLM-L-12-v2 cross encoder model

In [7]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2") #changed from all-MiniLM-L6-v2 and BAAI/bge-reranker-base
print("Cross encoder ready")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3571.72it/s]


Cross encoder ready


#### bm25 Search for keyword search

In [ ]:
def bm25_search(query, top_k=20):
    tokenized = bm25s.tokenize(query)                       
    results, scores = retriever.retrieve(tokenized, k=top_k) 
    output = []
    for i, score in zip(results[0], scores[0]):
        chunk = all_chunks[i].copy()
        chunk["bm25_score"] = float(score)
        chunk["chunk_id"] = str(chunk["chunk_id"])
        output.append(chunk)
    return output

#### Semantic search for dense vector

In [ ]:
def semantic_search(query, top_k=20):
    query_emb = bi_encoder.encode([query]).tolist()[0]   
    hits = qdrant.query_points(
        collection_name="papers",
        query=query_emb,
        limit=top_k
    ).points
    results = []
    for hit in hits:
        r = dict(hit.payload)
        r["section_title"] = r.pop("section", "UNKNOWN")
        r["chunk_id"] = str(r.get("chunk_id", ""))
        results.append(r)
    return results

#### Reciprocal Rank fusion to normalize Semantic results + BM25 indexing for better results

In [10]:
def reciprocal_rank_fusion(bm25_results, semantic_results, k=30):
    scores = {}
    chunk_map = {}
    for rank, result in enumerate(bm25_results):
        cid = str(result["chunk_id"])
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)
        chunk_map[cid] = result
    for rank, result in enumerate(semantic_results):
        cid = str(result["chunk_id"])
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank + 1)
        chunk_map[cid] = result
    sorted_ids = sorted(scores, key=lambda x: scores[x], reverse=True)
    results = []
    for cid in sorted_ids:
        chunk = chunk_map[cid].copy()
        chunk["rrf_score"] = scores[cid]
        results.append(chunk)
    return results

#### Hybrid search combining bm25s and semantic

In [11]:

def hybrid_search(query, top_k=50, fetch_k=100):
    bm25_results = bm25_search(query, top_k=fetch_k)
    semantic_results = semantic_search(query, top_k=fetch_k)
    fused = reciprocal_rank_fusion(bm25_results, semantic_results)
    return fused[:top_k]

#### Test for hybrid search

In [14]:
query = "what is swarm equilibrium density distribution boundary conditions"
hybrid_results = hybrid_search(query, top_k=20)
hybrid_results

[{'chunk_id': '817383855449daaf928de75431e6145b',
  'paper_id': 10,
  'text': 'we demonstrate different types of swarm equilibria via examples . in section \n [ sec : repulsive ] , we focus on purely repulsive endogenous interactions . \n we consider a bounded domain with no exogenous forces , a half - line subject to gravitational forces , and an unbounded domain subject to a quadratic exogenous potential , modeling attraction to a light , chemical , or nutrient source . \n for all three situations , we find exact solutions for swarm equilibria . \n for the first two examples , these equilibria consist of a density distribution that is classical in the interior of the domain , but contains [MATH]-functions at the boundaries . for the third example , the equilibrium is compactly supported with the density dropping discontinuously to zero at the edge of the support .',
  'section_title': 'ARTICLE',
  'rrf_score': 0.061669829222011384},
 {'chunk_id': '86480860ff7c16792aa1f822a587e234',
 

#### Rerank

In [15]:
def rerank(query, top_k_chunks, top_n=5):
    pairs = [(query, c["text"]) for c in top_k_chunks]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, top_k_chunks), key=lambda x: x[0], reverse=True)
    results = []
    for score, chunk in ranked[:top_n]:
        chunk = chunk.copy()
        chunk["rerank_score"] = float(score)
        results.append(chunk)
    return results

#### Test rerank

In [16]:
query = "what is swarm equilibrium density distribution boundary conditions"   
reranked = rerank(query, hybrid_results, top_n=5)

for r in reranked:
    print(f"RERANK: {r['rerank_score']:.4f}")
    print(f"SEC:    {r['section_title']}")
    print(f"TEXT:   {r['text'][:250]}")
    print("---")

RERANK: 2.6198
SEC:    ARTICLE
TEXT:   this energy can be interpreted as the continuum analog of the summed pairwise energy of the corresponding discrete ( particle ) model . 
 we will also exploit this energy to find equilibrium solutions and study their stability . in this paper 
 , we 
---
RERANK: 2.1184
SEC:    ARTICLE
TEXT:   we then derive an analogous continuum model and use variational methods to seek minimizers of its energy . 
 this process involves solution of a fredholm integral equation for the density . 
 for some choices of endogenous forces , we are able to fin
---
RERANK: 2.0673
SEC:    ARTICLE
TEXT:   we demonstrate different types of swarm equilibria via examples . in section 
 [ sec : repulsive ] , we focus on purely repulsive endogenous interactions . 
 we consider a bounded domain with no exogenous forces , a half - line subject to gravitation
---
RERANK: 2.0151
SEC:    ABSTRACT
TEXT:   we study equilibrium configurations of swarming biological organisms subject t

#### Load GROQ_QPI_KEY from .env

In [17]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
groq_client = Groq()

#### Calculate confidence score

In [18]:
def confidence_score(query, rank_results):
    if not rank_results:
        return 0.0, []
    scores = [r.get("rerank_score", r.get("rrf_score", 0.0)) for r in rank_results]
    min_s, max_s = min(scores), max(scores)
    if max_s > min_s:
        norm = [(s - min_s) / (max_s - min_s) for s in scores]
    else:
        norm = [1.0] * len(scores)
    top_score = norm[0]
    avg_score = np.mean(norm)
    dropoff   = norm[0] - norm[1] if len(norm) > 1 else 0
    confidence = 0.5 * top_score + 0.3 * avg_score + 0.2 * min(dropoff * 2, 1.0)
    return round(float(confidence), 4), scores

#### Use LLM Query routing to grade. LLM used - llama-3.1-8b-instant

In [19]:
def grading(query, rank_results, confidence):
    if not rank_results:
        return "irrelevant", "no result found"
    top_chunks = "\n\n--\n\n".join(r["text"][:300] for r in rank_results[:3])
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""You are a retrieval quality grader.
Query: {query}
Retrieved chunks:
{top_chunks}
Grade the retrieval. Respond in this exact format:
GRADE: <relevant|partial|irrelevant>
REASON: <one sentence>"""
        }]
    ) 
    text = response.choices[0].message.content.strip()
    grade_match = re.search(r"GRADE:\s*(relevant|partial|irrelevant)", text, re.IGNORECASE)
    reason_match = re.search(r"REASON:\s*(.+)", text)
    grade = grade_match.group(1).lower() if grade_match else "irrelevant"
    reason = reason_match.group(1).strip() if reason_match else "unknown"
    return grade, reason

#### Prompt engineer for better queries and try hybrid search with new prompt

In [20]:
def expand_query(query, n=2):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=150,
        messages=[{
            "role": "user",
            "content": f"""Generate {n} alternative search queries for a scientific paper retrieval system.
Original: {query}

Rules:
- Do NOT repeat or rephrase the original query
- Use different vocabulary — avoid repeating key terms from the original
- One must use broader conceptual language
- One must use related technical synonyms or adjacent concepts

Return ONLY the {n} new queries, one per line, no numbering, no original."""
        }]
    )
    lines = response.choices[0].message.content.strip().split("\n")
    variants = [l.strip() for l in lines if l.strip() and l.strip().lower() != query.lower()]
    return [query] + variants[:n]

def hybrid_search_expanded(query, top_k=50, fetch_k=100):
    queries = expand_query(query)
    seen, merged = set(), []
    for q in queries:
        for r in hybrid_search(q, top_k=fetch_k):
            cid = r["chunk_id"]
            if cid not in seen:
                seen.add(cid)
                merged.append(r)
    return merged[:top_k]

#### Retrieve files and grade them

In [21]:
def retrieve_and_grade(query, top_k=5, fetch_k=50, confidence_threshold=0.45):
    raw = hybrid_search_expanded(query, top_k=fetch_k)
    pre_confidence, _ = confidence_score(query, raw[:top_k])
    if pre_confidence >= confidence_threshold:
        results = raw[:top_k]
        confidence = pre_confidence
    else:
        results = rerank(query, raw, top_n=top_k)
        confidence, _ = confidence_score(query, results)
    grade, reason = grading(query, results, confidence)
    failed = (grade == "irrelevant" or confidence < confidence_threshold)
    return {
        "query": query,
        "results": results if not failed else [],
        "confidence": confidence,
        "grade": grade,
        "reason": reason,
        "retrieval_failed": failed,
        "fallback_message": "No relevant information found for this query." if failed else None
    }

#### Test retrieve and grade

In [22]:
queries = [
    "what is swarm equilibrium density distribution boundary conditions",  # should pass
    "what is the best recipe for pasta carbonara",                         # should fail
    "explain transformer attention mechanism",                             # borderline
]
for q in queries:
    result = retrieve_and_grade(q)
    print(f"\nQUERY:      {result['query']}")
    print(f"CONFIDENCE: {result['confidence']}")
    print(f"GRADE:      {result['grade']}")
    print(f"REASON:     {result['reason']}")
    print(f"FAILED:     {result['retrieval_failed']}")
    if result['retrieval_failed']:
        print(f"FALLBACK:   {result['fallback_message']}")
    else:
        print(f"TOP RESULT: {result['results'][0]['text'][:150]}")
    print("─" * 60)


QUERY:      what is swarm equilibrium density distribution boundary conditions
CONFIDENCE: 0.7867
GRADE:      partial
REASON:     The retrieved chunks contain some information related to the concept of "swarm equilibrium density distribution", including a mention of finding equilibrium solutions, but they do not specifically address the term "boundary conditions", which is the main query.
FAILED:     False
TOP RESULT: we demonstrate different types of swarm equilibria via examples . in section 
 [ sec : repulsive ] , we focus on purely repulsive endogenous interacti
────────────────────────────────────────────────────────────



QUERY:      what is the best recipe for pasta carbonara
CONFIDENCE: 0.7901
GRADE:      irrelevant
REASON:     The provided chunks discuss topics unrelated to pasta carbonara, such as fluid dynamics, quantum mechanics, and nonlinear normal modes, and have no relevance to a recipe.
FAILED:     True
FALLBACK:   No relevant information found for this query.
────────────────────────────────────────────────────────────



QUERY:      explain transformer attention mechanism
CONFIDENCE: 0.7739
GRADE:      partial
REASON:     The retrieved chunks are about concepts from physics and problem-solving in visual saliency modeling, but they do not mention the transformer attention mechanism, making them only partially relevant.
FAILED:     False
TOP RESULT: can be viewed conceptually as a net current [MATH] circulating around a primary coil . 
 circulating currents found high in the corona can be viewed a
────────────────────────────────────────────────────────────


#### Rewrite query 

In [23]:
def rewrite_query(original_query,grade,reason,attempt=1):
    strats = {
        1:"Rephrase the query using more technical/academic language",
        2:"Break the query into its core concept only. Make it short,concise and to the point",
        3:"Rewrite the query using different synonyms or related words"
    }
    strat = strats.get(attempt,strats[1])
    response = groq_client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": f"""You are a query rewriting assistant for a scientific paper retrieval system.

Original query: {original_query}
Retrieval grade: {grade}
Reason it failed: {reason}
Rewriting strategy: {strat}

Respond with ONLY the rewritten query, nothing else."""
        }]
    )
    return response.choices[0].message.content.strip()

In [24]:
RETRY_STRATEGIES = [
    ("hybrid",   lambda q: hybrid_search(q, top_k=50)),
    ("bm25",     lambda q: bm25_search(q, top_k=50)),
    ("semantic", lambda q: semantic_search(q, top_k=50)),
]

#### Retry mechanism. set max_retries to 2 to prevent overloading/slowing down

In [25]:
def retrieve_with_retry(query, max_retries=2, confidence_threshold=0.45):
    attempt_log = []
    current_query = query
    for attempt in range(max_retries + 1):
        strategy_name, search_fn = RETRY_STRATEGIES[min(attempt, len(RETRY_STRATEGIES) - 1)]
        print(f"\n[ATTEMPT {attempt}] strategy={strategy_name}")
        print(f"[QUERY]    {current_query}")

        raw_results = search_fn(current_query)
        pre_confidence, _ = confidence_score(current_query, raw_results[:5])

        if pre_confidence >= confidence_threshold:
            results = raw_results[:5]
            confidence = pre_confidence
        else:
            results = rerank(current_query, raw_results, top_n=5)
            confidence, _ = confidence_score(current_query, results)
        grade, reason = grading(current_query, results, confidence)
        failed = grade == "irrelevant" or confidence < confidence_threshold
        attempt_log.append({
            "attempt": attempt,
            "query": current_query,
            "strategy": strategy_name,
            "confidence": confidence,
            "grade": grade,
            "reason": reason,
            "failed": failed
        })
        print(f"[GRADE]    {grade} | confidence={confidence} | failed={failed}")
        if not failed:
            return {
                "query": query,
                "final_query": current_query,
                "results": results,
                "confidence": confidence,
                "grade": grade,
                "reason": reason,
                "retrieval_failed": False,
                "attempts": attempt_log
            }
        if attempt >= max_retries:
            break
        current_query = rewrite_query(query, grade, reason, attempt=attempt + 1)
        print(f"[REWRITE]  {current_query}")
    return {
        "query": query,
        "final_query": current_query,
        "results": [],
        "confidence": 0.0,
        "grade": "irrelevant",
        "reason": "all retry attempts failed",
        "retrieval_failed": True,
        "fallback_message": "I couldn't find relevant information after multiple attempts.",
        "attempts": attempt_log
    }

#### Test retry mechanism

In [91]:
test_queries = [
    "what is the best recipe for pasta carbonara",
    "nonlinear stability analysis of aggregation equations",
    "what is swarm equilibrium density distribution boundary conditions",
]
for q in test_queries:
    print("\n" + "═" * 60)
    print(f"ORIGINAL QUERY: {q}")
    print("═" * 60)
    result = retrieve_with_retry(q, max_retries=3)
    print(f"\n{'✅ SUCCESS' if not result['retrieval_failed'] else '❌ FAILED'}")
    print(f"FINAL QUERY:  {result['final_query']}")
    print(f"CONFIDENCE:   {result['confidence']}")
    print(f"GRADE:        {result['grade']}")
    print(f"ATTEMPTS:     {len(result['attempts'])}")
    if not result['retrieval_failed']:
        print(f"TOP RESULT:   {result['results'][0]['text'][:200]}")
    else:
        print(f"FALLBACK:     {result['fallback_message']}")

    print("\nATTEMPT LOG:")
    for a in result["attempts"]:
        status = "✅" if not a["failed"] else "❌"
        print(f"  {status} [{a['attempt']}] {a['strategy']:8} | conf={a['confidence']:.3f} | grade={a['grade']:12} | query: {a['query'][:60]}")


════════════════════════════════════════════════════════════
ORIGINAL QUERY: what is the best recipe for pasta carbonara
════════════════════════════════════════════════════════════

[ATTEMPT 0] strategy=hybrid
[QUERY]    what is the best recipe for pasta carbonara


[GRADE]    irrelevant | confidence=-8.2033 | failed=True
[REWRITE]  Query: "Optimizing the preparation protocol for a traditional Italian pasta dish, specifically 'carbonara', incorporating a review of extant formulations and identification of a consensus recipe."

[ATTEMPT 1] strategy=bm25
[QUERY]    Query: "Optimizing the preparation protocol for a traditional Italian pasta dish, specifically 'carbonara', incorporating a review of extant formulations and identification of a consensus recipe."


[GRADE]    partial | confidence=-5.8053 | failed=True
[REWRITE]  "pasta carbonara preparation protocol"

[ATTEMPT 2] strategy=semantic
[QUERY]    "pasta carbonara preparation protocol"
[GRADE]    relevant | confidence=-7.5504 | failed=True
[REWRITE]  What is the traditional method of preparing spaghetti with eggs and bacon.

[ATTEMPT 3] strategy=semantic
[QUERY]    What is the traditional method of preparing spaghetti with eggs and bacon.
[GRADE]    irrelevant | confidence=-8.7406 | failed=True

❌ FAILED
FINAL QUERY:  What is the traditional method of preparing spaghetti with eggs and bacon.
CONFIDENCE:   0.0
GRADE:        irrelevant
ATTEMPTS:     4
FALLBACK:     I couldn't find relevant information after multiple attempts.

ATTEMPT LOG:
  ❌ [0] hybrid   | conf=-8.203 | grade=irrelevant   | query: what is the best recipe for pasta carbonara
  ❌ [1] bm25     | conf=-5.805 | grade=partial      | query: Query: "Optimizing the preparation protocol for a traditiona
  ❌ [2] semantic | conf=-

[GRADE]    relevant | confidence=4.2569 | failed=False

✅ SUCCESS
FINAL QUERY:  nonlinear stability analysis of aggregation equations
CONFIDENCE:   4.2569
GRADE:        relevant
ATTEMPTS:     1
TOP RESULT:   a linear stability analysis is usually the first step of an analysis in this direction . for this 
 the method of interest is applied to a scalar linear test equation and stability conditions on metho

ATTEMPT LOG:
  ✅ [0] hybrid   | conf=4.257 | grade=relevant     | query: nonlinear stability analysis of aggregation equations

════════════════════════════════════════════════════════════
ORIGINAL QUERY: what is swarm equilibrium density distribution boundary conditions
════════════════════════════════════════════════════════════

[ATTEMPT 0] strategy=hybrid
[QUERY]    what is swarm equilibrium density distribution boundary conditions


[GRADE]    relevant | confidence=2.1061 | failed=False

✅ SUCCESS
FINAL QUERY:  what is swarm equilibrium density distribution boundary conditions
CONFIDENCE:   2.1061
GRADE:        relevant
ATTEMPTS:     1
TOP RESULT:   this energy can be interpreted as the continuum analog of the summed pairwise energy of the corresponding discrete ( particle ) model . 
 we will also exploit this energy to find equilibrium solutions

ATTEMPT LOG:
  ✅ [0] hybrid   | conf=2.106 | grade=relevant     | query: what is swarm equilibrium density distribution boundary cond


#### Run once to generated grounded truth benchmark based on the relevant ArXiv files

In [48]:
import random
def is_informative_chunk(chunk, min_words=30, max_math_ratio=0.15):
    text = chunk["text"] if isinstance(chunk, dict) else chunk
    tokens = text.split()
    if len(tokens) < min_words:
        return False
    math_tokens = sum(1 for t in tokens if '[MATH]' in t)
    return math_tokens / len(tokens) <= max_math_ratio

def generate_query_from_chunk(chunk_text):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=80,
        temperature=0.7,
        messages=[{
            "role": "user",
            "content": f"""You are helping build a retrieval benchmark for scientific papers.

Given this passage:
\"\"\"{chunk_text[:450]}\"\"\"

Write ONE natural search query that a researcher would type to find information like this.
- Use plain language, not a copy of the passage
- Do NOT use quotes or bullet points
- Respond with ONLY the query, nothing else"""
        }]
    )
    return response.choices[0].message.content.strip().strip('"').strip("'")

def build_grounded_benchmark(all_chunks, n=50, seed=42):
    random.seed(seed)
    pool = [c for c in all_chunks if is_informative_chunk(c)]
    print(f"usable chunks: {len(pool):,} / {len(all_chunks):,}")

    abstract_pool = [c for c in pool if c.get("section_title") == "ABSTRACT"]
    article_pool  = [c for c in pool if c.get("section_title") == "ARTICLE"]
    n_abstract = min(n // 2, len(abstract_pool))
    n_article  = min(n - n_abstract, len(article_pool))

    sampled = random.sample(abstract_pool, n_abstract) + random.sample(article_pool, n_article)
    random.shuffle(sampled)

    print(f"generating {len(sampled)} queries...")
    benchmark = []
    for idx, chunk in enumerate(sampled):
        try:
            query = generate_query_from_chunk(chunk["text"])
            benchmark.append({
                "id": idx,
                "query": query,
                "ground_truth_chunk_id": str(chunk["chunk_id"]),
                "ground_truth_paper_id": chunk["paper_id"],
                "source_section": chunk.get("section_title", "UNKNOWN"),
                "source_text_preview": chunk["text"][:200],
            })
            print(f"  [{idx+1}/{len(sampled)}] {query[:70]}")
        except Exception as e:
            print(f"  [{idx+1}/{len(sampled)}] failed: {e}")

    return benchmark

grounded_benchmark = build_grounded_benchmark(all_chunks, n=50)
with open("grounded_benchmark.json", "w") as f:
    json.dump(grounded_benchmark, f, indent=2)
print("saved to grounded_benchmark.json")

usable chunks: 422,488 / 443,080
generating 50 queries...
  [1/50] Quantum parameter influence in plasma dynamics
  [2/50] local density and energy calculation in molecular dynamics simulations
  [3/50] graphical model Bayesian learning two-step process
  [4/50] MACHOs in Andromeda galaxy detection survey
  [5/50] ultracompact x-ray binary pulsar optical pulsations
  [6/50] method of crosses and wrenches for finding bordering faces in polyhedr
  [7/50] entanglement degradation in non-inertial frames due to decoherence and
  [8/50] quantum coherent scattering atomic ensemble angular momentum measureme
  [9/50] columnar pins high temperature superconductors melting transition simu
  [10/50] Soft X-ray transient long-term optical observation
  [11/50] Sturm distance metric spaces mathematical definition
  [12/50] String topology applications to lagrangian embeddings in symplectic ma
  [13/50] galaxy virial factor inclinations effect on black hole mass
  [14/50] 2dF galaxy redshift survey 

#### Load benchmark in json format

In [49]:
with open("grounded_benchmark.json") as f:
    grounded_benchmark = json.load(f)
print(f"[INFO] Loaded {len(grounded_benchmark)} benchmark items")

[INFO] Loaded 50 benchmark items


#### Recall, MRR metrics for paper(soft) and exact(strict)

In [50]:
# exact = did we return the specific chunk the query came from
# paper = did we return any chunk from the right paper
def recall_at_k_exact(results, gt_chunk_id, k=5):
    return int(any(str(r["chunk_id"]) == gt_chunk_id for r in results[:k]))
def recall_at_k_paper(results, gt_paper_id, k=5):
    return int(any(r.get("paper_id") == gt_paper_id for r in results[:k]))
def mrr_exact(results, gt_chunk_id):
    for rank, r in enumerate(results, start=1):
        if str(r["chunk_id"]) == gt_chunk_id:
            return 1 / rank
    return 0.0
def mrr_paper(results, gt_paper_id):
    for rank, r in enumerate(results, start=1):
        if r.get("paper_id") == gt_paper_id:
            return 1 / rank
    return 0.0

#### BENCHMARK TO EVALUATE FULL PIPELINE

In [51]:
from collections import defaultdict

def run_grounded_eval(benchmark, top_k_retrieve=50, top_n_rerank=5):
    metrics = defaultdict(list)
    failed  = []
    for item in benchmark:
        query    = item["query"]
        gt_chunk = item["ground_truth_chunk_id"]
        gt_paper = item["ground_truth_paper_id"]
        section  = item["source_section"]
        try:
            raw = hybrid_search_expanded(query, top_k=top_k_retrieve)
            results = rerank(query, raw, top_n=top_n_rerank)
            r5_raw    = recall_at_k_exact(raw[:top_n_rerank], gt_chunk, k=top_n_rerank)
            r5_rerank = recall_at_k_exact(results,            gt_chunk, k=top_n_rerank)
            metrics["recall@5_raw"].append(r5_raw)
            metrics["recall@5_reranked"].append(r5_rerank)
            r5_exact  = recall_at_k_exact(results, gt_chunk, k=top_n_rerank)
            r5_pre    = recall_at_k_exact(raw,     gt_chunk, k=top_k_retrieve)
            r5_paper  = recall_at_k_paper(results, gt_paper, k=top_n_rerank)
            r50_paper = recall_at_k_paper(raw,     gt_paper, k=top_k_retrieve)
            metrics["recall@5_exact"].append(r5_exact)
            metrics["recall@5_paper"].append(r5_paper)
            metrics[f"recall@5_exact_{section}"].append(r5_exact)
            metrics[f"recall@5_paper_{section}"].append(r5_paper)
            metrics["recall@50_pre_exact"].append(r5_pre)
            metrics["recall@50_pre_paper"].append(r50_paper)
            metrics["mrr_exact"].append(mrr_exact(results, gt_chunk))
            metrics["mrr_paper"].append(mrr_paper(results, gt_paper))
            icon = "A" if r5_exact else ("B" if r5_paper else ("C" if r5_pre else "F"))
            print(f"{icon} [{section}] exact={r5_exact} paper={r5_paper} | {query[:60]}")
        except Exception as e:
            failed.append(query)
            print(f"[ERROR]error on: {query[:60]} — {e}")
    n_queries = len(benchmark) - len(failed)
    print(f"\n--- results ({n_queries} queries) ---")
    print(f"[INFO]recall@50 pre-rerank  (exact) : {np.mean(metrics['recall@50_pre_exact']):.3f}")
    print(f"[INFO]recall@5  post-rerank (exact) : {np.mean(metrics['recall@5_exact']):.3f}")
    print(f"[INFO]recall@5  post-rerank (paper) : {np.mean(metrics['recall@5_paper']):.3f}")
    print(f"[INFO]mrr exact                     : {np.mean(metrics['mrr_exact']):.3f}")
    print(f"[INFO]mrr paper                     : {np.mean(metrics['mrr_paper']):.3f}")

    for section in ["ABSTRACT", "ARTICLE"]:
        k = f"recall@5_exact_{section}"
        if metrics[k]:
            print(f"\n{section.lower()} ({len(metrics[k])} chunks)")
            print(f"  exact : {np.mean(metrics[k]):.3f}")
            print(f"  paper : {np.mean(metrics[f'recall@5_paper_{section}']):.3f}")
    true_lift = np.mean(metrics["recall@5_reranked"]) - np.mean(metrics["recall@5_raw"])
    print(f"\n[DEBUG]true reranker lift (reranked@5 vs raw@5) : {true_lift:+.3f}")
    if failed:
        print(f"\n[DEBUG]failed on {len(failed)} queries: {failed}")
    return metrics

In [52]:
eval_metrics = run_grounded_eval(grounded_benchmark, top_k_retrieve=50, top_n_rerank=5)

B [ARTICLE] exact=0 paper=1 | Quantum parameter influence in plasma dynamics


B [ABSTRACT] exact=0 paper=1 | local density and energy calculation in molecular dynamics s


A [ABSTRACT] exact=1 paper=1 | graphical model Bayesian learning two-step process


A [ABSTRACT] exact=1 paper=1 | MACHOs in Andromeda galaxy detection survey


A [ABSTRACT] exact=1 paper=1 | ultracompact x-ray binary pulsar optical pulsations


...

A [ABSTRACT] exact=1 paper=1 | satellite galaxy motion around isolated galaxies in the Sloa


A [ABSTRACT] exact=1 paper=1 | Distributed power allocation in cloud radio access networks 


A [ABSTRACT] exact=1 paper=1 | lozin's transformation effect on graph regularity


A [ARTICLE] exact=1 paper=1 | quantum potential in ground state of scattering process


A [ARTICLE] exact=1 paper=1 | charge disproportionation in triangular lattice materials

--- results (50 queries) ---
[INFO]recall@50 pre-rerank  (exact) : 0.880
[INFO]recall@5  post-rerank (exact) : 0.820
[INFO]recall@5  post-rerank (paper) : 0.900
[INFO]mrr exact                     : 0.663
[INFO]mrr paper                     : 0.830

abstract (25 chunks)
  exact : 0.920
  paper : 1.000

article (25 chunks)
  exact : 0.720
  paper : 0.800

[DEBUG]true reranker lift (reranked@5 vs raw@5) : +0.040


In [56]:
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.0)

In [59]:
def llm_route(query):
    prompt = f"""
You are a deterministic retrieval policy classifier. Your sole function is to map an incoming query to the optimal retrieval configuration.

Available retrieval strategies:
1. bm25: Best for exact terminology, acronyms, specific codes, or exact paper/entity names.
2. dense: Best for semantic similarity, conceptual questions, and natural language phrasing.
3. hybrid: Best for mixed queries containing both specific keywords and broader concepts.
4. hybrid_rerank: Best for difficult, complex, or multi-hop questions requiring high precision.

[CRITICAL BEHAVIORAL CONSTRAINTS]
- Do NOT output any free-form reasoning, thinking blocks, or chain-of-thought.
- Do NOT output any explanations, introductions, or postscripts.
- You must behave strictly as a token-optimized classifier.
- Your output must be ONLY valid, minified JSON matching the schema below.

[OUTPUT SCHEMA]
{{
  "strategy": "bm25" | "dense" | "hybrid" | "hybrid_rerank",
  "top_k": integer,
  "rerank": boolean
}}

Input Query: {query}
JSON Output:
"""
    response = llm.invoke([prompt])
    raw = response.content.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON found in LLM response: {raw!r}")
        return json.loads(match.group())

In [62]:
def generate_answer(query, results, top_k=3):
    context = "\n\n---\n\n".join(r["text"][:400] for r in results[:top_k])
    
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""You are a scientific research assistant. Answer the query using only the provided context.

Context:
{context}

Query: {query}

Rules:
- Answer concisely in 2-3 sentences
- Only use information from the context
- If the context doesn't contain the answer, say so

Answer:"""
        }]
    )
    return response.choices[0].message.content.strip()

#### EVALUATING GENERATION

In [ ]:
def run_generation_eval(benchmark, top_k_retrieve=50, top_n_rerank=5, n=10):
    results_log = []
    sample = random.sample(benchmark, min(n, len(benchmark)))
    for item in sample:
        query = item["query"]
        gt_chunk = item["ground_truth_chunk_id"]
        gt_paper = item["ground_truth_paper_id"]
        route = llm_route(query)
        if route["strategy"] == "bm25":
            results = bm25_search(query, top_k=top_k_retrieve)
        elif route["strategy"] == "dense":
            results = semantic_search(query, top_k=top_k_retrieve)
        elif route["strategy"] == "hybrid":
            results = hybrid_search_expanded(query, top_k=top_k_retrieve)
        else:
            results = retrieve_with_retry(query)
        reranked = rerank(query, results, top_n=top_n_rerank)
        answer = generate_answer(query, reranked)
        confidence = reranked[0].get("rrf_score", 1.0) if reranked else 0.0
        grade, reason = grading(query, reranked, confidence)
        results_log.append({
            "query": query,
            "strategy": route["strategy"],
            "grade": grade,
            "reason": reason,
            "answer": answer,
            "gt_chunk": gt_chunk,
            "gt_paper": gt_paper,
        })
        icon = "✅" if grade == "relevant" else ("⚠️" if grade == "partial" else "❌")
        print(f"{icon} [{grade}] {query[:60]}")
        print(f"   → {answer[:120]}")
        print(f"   reason: {reason}\n")
    grades = [r["grade"] for r in results_log]
    print("--- generation eval summary ---")
    print(f"relevant  : {grades.count('relevant')}/{len(grades)}")
    print(f"partial   : {grades.count('partial')}/{len(grades)}")
    print(f"irrelevant: {grades.count('irrelevant')}/{len(grades)}")
    return results_log
results_log = run_generation_eval(grounded_benchmark, n=10)

✅ [relevant] 2dF galaxy redshift survey dynamical mass analysis
   → We compute the dynamical mass, [MATH], for 809 isolated host galaxies in the 100k data release of the 2dF galaxy redshif
   reason: The retrieved chunks include sentences directly related to the topic of dynamical mass analysis of the 2Df galaxy redshift survey.



✅ [relevant] relativistic jet formation in M87 black holes
   → The accretion of matter onto a massive black hole is believed to feed the relativistic plasma jets found in active galac
   reason: The retrieved chunks all directly relate to the topic of relativistic jet formation in M87 black holes, mentioning its connection to the accretion of matter onto a massive black hole in active galactic nuclei.



✅ [relevant] satellite galaxy motion around isolated galaxies in the Sloa
   → The motion of satellite galaxies around normal galaxies, particularly at distances 50 - 500 kpc, can provide a sensitive
   reason: The retrieved chunks of text mention concepts and data (e.g., Sloan Digital Sky Survey, isolated host galaxies, dynamical mass) related to the study of satellite galaxy motion around isolated galaxies.

✅ [relevant] Bayesian inference for spectrum sensing in cognitive radio s
   → For spectrum sensing in cognitive radio systems, a criterion from a Bayesian perspective is mentioned as "minimum Bayesi
   reason: The retrieved chunks describe the importance of reliable spectrum sensing and detection in cognitive radio systems, mentioning specific techniques and considerations, which is relevant to the topic of Bayesian inference for spectrum sensing.

✅ [relevant] charge disproportionation in triangular lattice materials
   → In the context of triangular lattice materials, charge d

✅ [relevant] black hole simulation boundary conditions
   → The black hole boundary conditions used are the black hole boundary conditions (BHBC) or the quasi-boundary conditions (
   reason: The retrieved chunks provide information regarding the setup of black hole simulations, including boundary conditions and specific parameters used in the 3D simulation.



✅ [relevant] spin polarization in materials with Rashba effect and Yang M
   → Spin polarization is presented in a quite general form using an effective Yang-Mills field, with time and space derivati
   reason: The retrieved chunks explicitly mention key terms such as Rashba effect, Yang-Mills field, and spin-orbit interaction, indicating that they are highly relevant to the query topic.

✅ [relevant] x-ray emission in ngc 4151 galaxy inner region
   → The x-ray emission in the inner region of the NGC 4151 galaxy was studied within a radius of approximately [MATH] pc. Hi
   reason: The retrieved chunks contain information about the X-ray emission in the NGC 4151 galaxy's inner region, including the use of Chandra observations and spectral analysis.

✅ [relevant] detection of warm hot intragroup medium in galaxy clusters
   → The warm-hot intragroup medium in galaxy clusters has thus far evaded direct detection. As a result, its density and tem
   reason: The retrieved chunks provide in

#### Testing generation quality

In [67]:
query = "how to get a job in 2026"
route = llm_route(query)
if route["strategy"] == "bm25":
    results = bm25_search(query)

elif route["strategy"] == "dense":
    results = semantic_search(query)

elif route["strategy"] == "hybrid":
    results = hybrid_search_expanded(query)
else:
    results = retrieve_with_retry(query)
answer = generate_answer(query,results)
answer

'To get a job in 2026, following your passion and studying something you\'re interested in might be a good approach, as hinted at by the phrase "follow your passion" and the mention of being lucky when a skill evolves into being marketable. No specific steps are provided in the given context, only general advice.'

In [68]:
query = "how to make pasta"
route = llm_route(query)
ans = generate_answer(query,results)
ans

'The provided context does not contain information on how to make pasta, so I cannot provide an answer.'